In [30]:
from docx import Document
from transformers import AutoTokenizer

# === Шаг 1. Извлечь жирные сущности и текст ===
def extract_bold_entities(docx_path):
    doc = Document(docx_path)
    sentences = []
    bold_entities = set()

    for para in doc.paragraphs:
        sentence = ""
        for run in para.runs:
            text = run.text
            if run.bold and text.strip():
                bold_entities.add(text.strip())
            sentence += text
        if sentence.strip():
            sentences.append(sentence.strip())
    return sentences, bold_entities

# === Шаг 2. Загрузить файл с сущностями и категориями ===
def load_entity_labels(file_path):
    entity_to_label = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().rsplit(' ', 1)
            if len(parts) == 2:
                entity, label = parts
                entity_to_label[entity.strip()] = label.strip()
    return entity_to_label

# === Шаг 3. Связать жирные сущности с категориями ===
def match_entities_with_labels(bold_entities, entity_to_label):
    matched = {}
    unmatched = []

    for entity in bold_entities:
        if entity in entity_to_label:
            matched[entity] = entity_to_label[entity]
        else:
            unmatched.append(entity)
    return matched, unmatched

# === Шаг 4. Основной код ===
if __name__ == "__main__":
    docx_path = "/home/chupchik/voinaIMir/texts/вим-1-том-2-часть-главы-1-12ЛЕВ (2).docx"      # <-- замени на путь к своему .docx
    labels_path = "/home/chupchik/voinaIMir/entities (2).txt" # <-- замени на путь к .txt с категориями

    # Загружаем
    sentences, bold_entities = extract_bold_entities(docx_path)
    entity_to_label = load_entity_labels(labels_path)
    matched, unmatched = match_entities_with_labels(bold_entities, entity_to_label)

    # Показываем результат
    print("\n📌 Найденные сущности с категориями:")
    for entity, label in matched.items():
        print(f"  {entity} → {label}")

    if unmatched:
        print("\n Сущности, для которых НЕ найдены категории:")
        for entity in unmatched:
            print(f"  {entity}")

    print("\n📄 Всего предложений:", len(sentences))
    print("Пример предложений:")
    for i, s in enumerate(sentences[:5]):
        print(f"  {i+1}: {s}")



📌 Найденные сущности с категориями:
  Несвицкого → PER
  князем Багратионом → PER
  Миронов → PER
  Подольский → ORG
  Наполеону → PER
  Никитенко → PER
  Вейротером → PER
  Лавг’ушка → PER
  Польше → GPE
  эрцгерцог Карл → PER
  венско-цнаймскую → FAC
  Амштетене → GPE
  Тулон → GPE
  Ланна → PER
  Ипполиту → PER
  Каменскиим-отцом → PER
  Князь Ипполит → PER
  Телянину → PER
  Тушина → PER
  Шенграбена → GPE
  Лемарруа (Lemarrois) → PER
  Васьки Денисова → PER
  Дунаю → LOC
  Kaiser Alexander → PER
  Ауэрспергом → PER
  Несвицкий → PER
  Вася → PER
  Дунаем → LOC
  Bonaparte → PER
  Ауэрсперга → PER
  Бандарчука → PER
  венский мост → FAC
  Наполеон → PER
  Денисовым → PER
  Ипполит → PER
  императора Александра → PER
  Бонапарта → PER
  Франц → PER
  Князю Андрею → PER
  Тимохина → PER
  Брунове → GPE
  Кутузовскою → PER
  Шенграбен → GPE
  Быкова → PER
  Францией → GPE
  Богемские → LOC
  Телянин → PER
  императору Францу → PER
  Козловского → GPE
  Маком → PER
  павлоградскому → 

In [35]:
import os
import random
import re
from pathlib import Path
from docx import Document
from transformers import AutoTokenizer

# === Пути ===
docx_path = "/home/chupchik/voinaIMir/texts/вим-1-том-2-часть-главы-1-12ЛЕВ (2).docx"
labels_path = "/home/chupchik/voinaIMir/entities (2).txt"
output_dir = Path("/home/chupchik/voinaIMir/rubert.ipynb/datasets")
output_dir.mkdir(parents=True, exist_ok=True)

# === Шаг 1: Извлечение жирных сущностей и текста ===
def extract_bold_entities(docx_path):
    doc = Document(docx_path)
    sentences = []
    bold_entities = set()

    for para in doc.paragraphs:
        sentence = ""
        for run in para.runs:
            text = run.text
            if run.bold and text.strip():
                bold_entities.add(text.strip())
            sentence += text

        # Разбиваем абзац на предложения
        if sentence.strip():
            split_sentences = re.split(r'(?<=[.!?])\s+', sentence.strip())
            sentences.extend(split_sentences)

    return sentences, bold_entities

# === Шаг 2: Загрузка категорий сущностей ===
def load_entity_labels(file_path):
    entity_to_label = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().rsplit(' ', 1)
            if len(parts) == 2:
                entity, label = parts
                entity_to_label[entity.strip()] = label.strip()
    return entity_to_label

# === Шаг 3: BIO-разметка с учетом offset-ов ===
def tokenize_and_tag_sentences(sentences, matched_entities, tokenizer):
    data = []
    for sentence in sentences:
        encoding = tokenizer(
            sentence,
            return_offsets_mapping=True,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors=None,
        )

        input_ids = encoding["input_ids"]
        offsets = encoding["offset_mapping"]
        tokens = tokenizer.convert_ids_to_tokens(input_ids)
        labels = ["O"] * len(tokens)

        for entity, tag in matched_entities.items():
            if not entity.strip():
                continue
            start_idx = 0
            while True:
                start = sentence.find(entity, start_idx)
                if start == -1:
                    break
                end = start + len(entity)

                for i, (token_start, token_end) in enumerate(offsets):
                    if token_start is None or token_end is None:
                        continue
                    if token_start >= start and token_end <= end:
                        if labels[i] == "O":
                            labels[i] = f"B-{tag}" if token_start == start else f"I-{tag}"
                start_idx = end

        data.append((tokens, labels))
    return data

# === Поддержка: объединение субтокенов ===
def merge_subword_tokens(tokens, labels):
    merged_tokens = []
    merged_labels = []

    for token, label in zip(tokens, labels):
        if token.startswith("##") and merged_tokens:
            merged_tokens[-1] += token[2:]
        else:
            merged_tokens.append(token)
            merged_labels.append(label)
    return merged_tokens, merged_labels

# === Шаг 4: Сохранение в .txt ===
def save_to_txt(data, path, tokenizer):
    with open(path, "w", encoding="utf-8") as f:
        for tokens, labels in data:
            tokens, labels = merge_subword_tokens(tokens, labels)
            for t, l in zip(tokens, labels):
                if t in tokenizer.all_special_tokens:
                    continue
                f.write(f"{t} {l}\n")
            f.write("\n")  # Пустая строка между предложениями

# === Шаг 5: Основной запуск ===
if __name__ == "__main__":
    # Загрузка текста и сущностей
    sentences, bold_entities = extract_bold_entities(docx_path)
    entity_to_label = load_entity_labels(labels_path)

    matched = {e: entity_to_label[e] for e in bold_entities if e in entity_to_label}
    tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

    print(f"📄 Найдено предложений: {len(sentences)}")
    print(f"🏷  Жирных сущностей с категорией: {len(matched)}")

    # BIO-разметка
    data = tokenize_and_tag_sentences(sentences, matched, tokenizer)

    # Перемешиваем и делим
    random.seed(42)
    random.shuffle(data)
    n = len(data)
    train_data = data[:int(n * 0.7)]
    val_data = data[int(n * 0.7):int(n * 0.85)]
    test_data = data[int(n * 0.85):]

    # Сохраняем
    save_to_txt(train_data, output_dir / "train.txt", tokenizer)
    save_to_txt(val_data, output_dir / "val.txt", tokenizer)
    save_to_txt(test_data, output_dir / "test.txt", tokenizer)

    print("\n✅ Готово! BIO-разметка сохранена в:")
    print(f" - {output_dir / 'train.txt'}")
    print(f" - {output_dir / 'val.txt'}")
    print(f" - {output_dir / 'test.txt'}")


/home/chupchik/voinaIMir/.venv/lib/python3.8/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


📄 Найдено предложений: 2456
🏷  Жирных сущностей с категорией: 304

✅ Готово! BIO-разметка сохранена в:
 - /home/chupchik/voinaIMir/rubert.ipynb/datasets/train.txt
 - /home/chupchik/voinaIMir/rubert.ipynb/datasets/val.txt
 - /home/chupchik/voinaIMir/rubert.ipynb/datasets/test.txt


In [36]:
import os
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import DatasetDict, Dataset
from sklearn.metrics import classification_report
from seqeval.metrics import f1_score, classification_report as seqeval_classification_report

# === Пути ===
data_dir = Path("/home/chupchik/voinaIMir/rubert.ipynb/datasets")
model_checkpoint = "DeepPavlov/rubert-base-cased"
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-GPE', 'I-GPE', 'B-FAC', 'I-FAC', 'B-VEH', 'I-VEH']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# === Загрузка BIO-данных из файлов ===
def load_bio_dataset(data_dir):
    def read_file(file_path):
        tokens = []
        labels = []
        with open(file_path, encoding='utf-8') as f:
            tok_seq = []
            lab_seq = []
            for line in f:
                line = line.strip()
                if not line:
                    if tok_seq:
                        tokens.append(tok_seq)
                        labels.append(lab_seq)
                        tok_seq, lab_seq = [], []
                else:
                    token, label = line.split()
                    tok_seq.append(token)
                    lab_seq.append(label)
        return {"tokens": tokens, "ner_tags": labels}

    return DatasetDict({
        "train": Dataset.from_dict(read_file(data_dir / "train.txt")),
        "validation": Dataset.from_dict(read_file(data_dir / "val.txt")),
        "test": Dataset.from_dict(read_file(data_dir / "test.txt")),
    })

# === Токенизатор ===
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# === Преобразуем строки меток в ID ===
def encode_labels(example):
    tokens = example["tokens"]
    labels = example["ner_tags"]
    tokenized = tokenizer(tokens, is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[labels[word_idx]])
        else:
            # В случае субтокена
            label = labels[word_idx]
            if label.startswith("B-"):
                label = label.replace("B-", "I-")
            aligned_labels.append(label2id[label])
        previous_word_idx = word_idx
    tokenized["labels"] = aligned_labels
    return tokenized

# === Загружаем данные ===
dataset = load_bio_dataset(data_dir)
encoded_dataset = dataset.map(encode_labels, batched=False)

# === Модель ===
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# === Аргументы тренировки ===
training_args = TrainingArguments(
    output_dir="./rubert-ner",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

# === Вычисление метрик ===
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        cur_labels = []
        cur_preds = []
        for p, l in zip(pred, lab):
            if l != -100:
                cur_labels.append(id2label[l])
                cur_preds.append(id2label[p])
        true_labels.append(cur_labels)
        true_predictions.append(cur_preds)

    f1 = f1_score(true_labels, true_predictions)
    print(seqeval_classification_report(true_labels, true_predictions))
    return {"f1": f1}

# === Обучение ===
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./rubert-ner/final")
tokenizer.save_pretrained("./rubert-ner/final")


100%|██████████| 369/369 [00:00<00:00, 3083.17ex/s]
Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertForTokenClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.decoder.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weig

{'loss': 0.3735, 'learning_rate': 1.8837209302325582e-05, 'epoch': 0.23}


 12%|█▏        | 100/860 [02:46<21:06,  1.67s/it]

{'loss': 0.0842, 'learning_rate': 1.7674418604651163e-05, 'epoch': 0.47}


 17%|█▋        | 150/860 [04:09<17:59,  1.52s/it]

...

100%|██████████| 860/860 [25:05<00:00,  1.82s/it]

              precision    recall  f1-score   support

         FAC       0.00      0.00      0.00         5
         GPE       0.74      0.80      0.77        25
         LOC       0.67      0.67      0.67         3
         ORG       0.50      0.50      0.50         2
         PER       0.95      0.94      0.94       191
         VEH       0.00      0.00      0.00         3

   micro avg       0.89      0.89      0.89       229
   macro avg       0.48      0.48      0.48       229
weighted avg       0.88      0.89      0.89       229

{'eval_loss': 0.0557544082403183, 'eval_f1': 0.888402625820569, 'eval_runtime': 16.6652, 'eval_samples_per_second': 22.082, 'eval_steps_per_second': 2.76, 'epoch': 4.0}


100%|██████████| 860/860 [25:08<00:00,  1.75s/it]


{'train_runtime': 1508.6226, 'train_samples_per_second': 4.558, 'train_steps_per_second': 0.57, 'train_loss': 0.04425591551373864, 'epoch': 4.0}


('./rubert-ner/final/tokenizer_config.json',
 './rubert-ner/final/special_tokens_map.json',
 './rubert-ner/final/vocab.txt',
 './rubert-ner/final/added_tokens.json',
 './rubert-ner/final/tokenizer.json')

In [50]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Загрузка токенизатора и модели
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
model = AutoModelForTokenClassification.from_pretrained("./rubert-ner/final")  # Укажите путь к вашей модели

# Новый текст для предсказания
new_sentences = [
    "Князь Андрей и Наташа Ростова молчал, и лицо его так было неприятно, что Пьер обращался более к добродушному батальонному командиру Тимохину, чем к Болконскому.",
    "29-го мая Наполеон выехал из Дрездена, где он пробыл три недели, окруженный двором, составленным из принцев, герцогов, королей и даже одного императора.",
    "Кутузов сидел, понурив седую голову и опустившись тяжелым телом, на покрытой ковром лавке, на том самом месте, на котором утром его видел Пьер. Он не делал никаких распоряжении, а только соглашался или не соглашался на то, что предлагали ему."
]

# Токенизация предложений
inputs = tokenizer(new_sentences, padding=True, truncation=True, return_tensors="pt")

# Получение предсказаний (выход модели)
with torch.no_grad():  # Отключаем вычисление градиентов
    outputs = model(**inputs)
    predictions = outputs.logits

# Получаем индексы наиболее вероятных меток
predicted_labels = torch.argmax(predictions, dim=-1)

# Преобразуем индексы в метки
label_map = model.config.id2label  # Сопоставление индекса метки и её имени
predicted_labels = predicted_labels.cpu().numpy()

# Печатаем результаты
for i, (sentence, label_seq) in enumerate(zip(new_sentences, predicted_labels)):
    # Преобразуем id в токены для текущего предложения
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][i])
    
    print(f"Предсказания для: {sentence}")
    for token, label in zip(tokens, label_seq):
        if token.startswith("##"):  # Пропускаем субтокены
            continue
        print(f"{token} => {label_map[label]}")
    print()  # Пустая строка между предложениями


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Предсказания для: Князь Андрей и Наташа Ростова молчал, и лицо его так было неприятно, что Пьер обращался более к добродушному батальонному командиру Тимохину, чем к Болконскому.
[CLS] => O
Князь => B-PER
Андрей => I-PER
и => O
Наташа => B-PER
Ростова => I-PER
молчал => O
, => O
и => O
лицо => O
его => O
так => O
было => O
неприят => O
, => O
что => O
Пьер => B-PER
обращался => O
более => O
к => O
добродуш => O
батальон => O
командиру => O
Тимо => B-PER
, => O
чем => O
к => O
Бол => B-PER
. => O
[SEP] => O
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => B-PER
[PAD] => B-PER
[PAD] => I-PER
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => O
[PAD] => B-PER
[PAD] => I-PER
[PAD] => B-PER
[PAD] => O
[PAD] => O
[PAD] => B-PER

Предсказания для: 29-го мая Наполеон выехал из Дрездена, где он пробыл три недели, окруженный двором, составленным из принцев, герцогов, королей и даже одного императора.
[CLS] => O
29 => O
- => O
го => O
мая => O
Наполеон => B-PER
выехал => O
из => O
Дрездена => B-GPE
, 